In [ ]:
import torch
import torch.nn as nn # Not strictly needed if T5Simplifier doesn't define new layers, but harmless.
import torch.nn.functional as F # Not strictly needed here, but harmless.
from transformers import T5Tokenizer, T5ForConditionalGeneration

class T5Simplifier:
    """
    An agent responsible for simplifying extracted sentences using a pre-trained T5 model.
    It can process single sentences or batches of sentences for efficiency.
    """
    def __init__(self, model_name: str = 't5-small', device: torch.device = None,
                 max_input_len: int = 512, max_output_len: int = 128):
        """
        Initializes the T5Simplifier.

        Args:
            model_name (str): Name of the pre-trained T5 model to load (e.g., 't5-small', 't5-base').
            device (torch.device, optional): The device to load the model onto (e.g., 'cuda', 'cpu').
                                             Defaults to 'cuda' if available, else 'cpu'.
            max_input_len (int): Maximum token length for the input text.
            max_output_len (int): Maximum token length for the simplified output.
        """
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"T5Simplifier will use device: {self.device}")

        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name).to(self.device)

        self.max_input_len = max_input_len
        self.max_output_len = max_output_len

    def simplify(self, input_text: str, max_length: int = None) -> str:
        """
        Simplifies a single input text using the T5 model.

        Args:
            input_text (str): The text (e.g., an extracted sentence) to be simplified.
            max_length (int, optional): Max length for the output sequence.
                                        Overrides self.max_output_len if provided.

        Returns:
            str: The simplified version of the input text.
        """
        # Define the generation max_length, prioritizing argument over self.max_output_len
        actual_max_output_len = max_length if max_length is not None else self.max_output_len

        # Prefix helps T5 understand the task type. Clean input by stripping and replacing newlines.
        prefixed_input = "simplify: " + input_text.strip().replace("\n", " ")

        # Encode the input text, applying truncation and max_length for the input tokens
        inputs = self.tokenizer(
            prefixed_input,
            return_tensors='pt',
            truncation=True,
            max_length=self.max_input_len # Explicitly setting max_length for input tokenization
        ).to(self.device)

        # Generate the simplified output
        outputs = self.model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'], # Crucial for generation
            max_length=actual_max_output_len,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2 # Added to potentially improve output quality by reducing repetition
        )

        # Decode the generated tokens back to a string
        simplified_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return simplified_text

    def simplify_batch(self, input_texts: list[str], max_length: int = None) -> list[str]:
        """
        Simplifies a batch of input texts using the T5 model.

        Args:
            input_texts (list[str]): A list of texts to be simplified.
            max_length (int, optional): Max length for the output sequence for each text.
                                        Overrides self.max_output_len if provided.

        Returns:
            list[str]: A list of simplified versions of the input texts.
        """
        actual_max_output_len = max_length if max_length is not None else self.max_output_len

        # Apply prefix and clean each text in the batch
        prefixed_inputs = ["simplify: " + text.strip().replace("\n", " ") for text in input_texts]

        # Encode the batch of texts. padding=True will pad to the longest sequence in the batch.
        inputs = self.tokenizer(
            prefixed_inputs,
            return_tensors='pt',
            padding=True, # Pad all sequences in the batch to the same length
            truncation=True,
            max_length=self.max_input_len
        ).to(self.device)

        # Generate outputs for the entire batch
        outputs = self.model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'], # Pass attention mask for batched generation
            max_length=actual_max_output_len,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        # Decode all generated outputs
        simplified_texts = [self.tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        return simplified_texts


# Sample usage for testing
if __name__ == "__main__":
    print("--- Initializing T5Simplifier ---")
    # You can change model_name to 't5-base' for potentially better quality (more VRAM needed)
    simplifier = T5Simplifier(model_name='t5-small')
    print(f"Simplifier initialized on {simplifier.device}")

    # --- Test single sentence simplification ---
    print("\n--- Testing Single Sentence Simplification ---")
    extracted_complex_sentence = "The labyrinthine complexities of quantum entanglement posit a reality where causality may be non-local."
    simplified_single = simplifier.simplify(extracted_complex_sentence)
    print("Original (single):", extracted_complex_sentence)
    print("Simplified (single):", simplified_single)
    # Expected output similar to: Quantum entanglement is complicated. It says cause and effect may not be local.

    # --- Test batch sentence simplification ---
    print("\n--- Testing Batch Sentence Simplification ---")
    extracted_complex_sentences = [
        "The recent geopolitical turmoil exacerbated the already precarious economic stability of developing nations.",
        "Despite the inherent difficulties, the nascent artificial intelligence community evinced remarkable resilience.",
        "A protracted period of atmospheric dryness frequently culminates in widespread agricultural depredation."
    ]
    simplified_batch = simplifier.simplify_batch(extracted_complex_sentences)
    print("Original (batch):")
    for i, text in enumerate(extracted_complex_sentences):
        print(f"  [{i+1}] {text}")
    print("\nSimplified (batch):")
    for i, text in enumerate(simplified_batch):
        print(f"  [{i+1}] {text}")

    # Expected output similar to:
    # [1] Recent political unrest made poor countries' economies unstable.
    # [2] The new AI community showed great strength despite problems.
    # [3] Long dry spells often lead to widespread crop damage.

--- Initializing T5Simplifier ---
T5Simplifier will use device: cpu


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Simplifier initialized on cpu

--- Testing Single Sentence Simplification ---
Original (single): The labyrinthine complexities of quantum entanglement posit a reality where causality may be non-local.
Simplified (single): Einfach simpli: Die labyrinthine complexities of quantum entanglement posit a reality where causality may be non-local.

--- Testing Batch Sentence Simplification ---
Original (batch):
  [1] The recent geopolitical turmoil exacerbated the already precarious economic stability of developing nations.
  [2] Despite the inherent difficulties, the nascent artificial intelligence community evinced remarkable resilience.
  [3] A protracted period of atmospheric dryness frequently culminates in widespread agricultural depredation.

Simplified (batch):
  [1] Verein simpli simpli: Die recent geopolitical turmoil a exacerbiert die bereits precarious economic stability der Entwicklung nations.
  [2] : Despite the inherent difficulties, the nascent artificial intelligence communit